# Explore HMRC OTS trade data before the full pull

Scratch notebook (not logged, nothing saved): check what the fields mean and how big the pull
for all minerals in `config/minerals.toml` would be, for a given start month.

In [ ]:
import time
import tomllib
from pathlib import Path

import pandas as pd
import requests

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE = "https://api.uktradeinfo.com"

with open(REPO_ROOT / "config" / "minerals.toml", "rb") as f:
    minerals = tomllib.load(f)["minerals"]

def get_json(path, params=None):
    """GET one page of JSON from the HMRC API (enough for exploration)."""
    response = requests.get(f"{BASE}/{path}", params=params, timeout=60)
    response.raise_for_status()
    return response.json()

In [ ]:
# What do the FlowTypeId values mean?
pd.DataFrame(get_json("FlowType")["value"])

In [ ]:
# One code (lithium carbonate), recent months: how many rows, and how much comes from the port split?
sample = pd.DataFrame(get_json("OTS", {"$filter": "CommodityId eq 28369100 and MonthId ge 202401"})["value"])
print(sample.shape)
print("ports:", sample["PortId"].nunique(),
      "| countries:", sample["CountryId"].nunique(),
      "| flows:", sample["FlowTypeId"].nunique())
sample.head(10)

In [ ]:
START_MONTH = 201501  # change to compare start years

def count_rows(codes, start=START_MONTH, batch=10):
    """Ask the API how many OTS rows match, without downloading them (in batches of codes)."""
    total = 0
    for i in range(0, len(codes), batch):
        codes_filter = " or ".join(f"CommodityId eq {int(c)}" for c in codes[i:i + batch])
        response = requests.get(f"{BASE}/OTS/$count",
                                params={"$filter": f"({codes_filter}) and MonthId ge {start}"},
                                timeout=120)
        response.raise_for_status()
        total += int(response.content.decode("utf-8-sig"))  # utf-8-sig drops the BOM the API adds
        time.sleep(0.5)  # be gentle with the API
    return total

counts = {}
for key, m in minerals.items():
    counts[key] = count_rows(m["cn8"])
    print(f"{key}: {counts[key]:,}")
    time.sleep(0.5)  # be gentle with the API

counts = pd.Series(counts).sort_values(ascending=False)
print(f"\ntotal rows from {START_MONTH}: {counts.sum():,}")
counts